# 6단계 — 결론 정리

> 계획.md 6단계 스펙(성능 요약표 → 공식 모델과 비교 → 한계점 → v2 방향)에 따라 프로젝트를 마무리한다.
> 모든 수치는 4단계 평가(`results/run3/test_metrics.json`, `results/run3/test_classification_report.txt`)와
> 3단계 실험 기록(진행일지.md)에서 그대로 가져온 것이며, 새로 계산한 값은 없다.

## 1. 성능 요약표 (test셋 9,026건 기준, Run3 모델)

| Entity | Precision | Recall | F1 | support |
|--------|-----------|--------|----|----|
| PER | 0.61 | 0.50 | 0.55 | 269 |
| LOC | 0.76 | 0.89 | 0.82 | 291 |
| ORG | 0.90 | 0.94 | 0.92 | 6,573 |
| **전체 (micro avg)** | 0.89 | 0.92 | **0.9045** | 7,133 |

목표 F1 0.80을 확실히 초과 달성했다 (0.9045). val F1(0.9077)과 거의 동일해 과적합도 없다.
다만 이 micro F1은 ORG가 support의 92%를 차지하는 데서 오는 착시가 크다 — PER(0.55)은
목표치보다는 높지만 ORG 대비 뚜렷하게 약하고, 3단계에서 두 차례(Run4, Run5) class weight로
개선을 시도했으나 모두 실패했다 (아래 한계점 참조).

## 2. 공식 모델과 비교

| 모델 | 파라미터 | F1 | 비고 |
|------|---------|-----|------|
| GPT-OSS-120B (QLoRA) | 120B | **0.9589** | AI Hub 공식 벤치마크, 비공개 test셋(2,100건) 기준 |
| KLUE-BERT (본 프로젝트, Run3) | 110M | **0.9045** | Validation에서 직접 분리한 test셋(9,026건) 기준 |

차이(0.0544)의 원인은 한 가지가 아니라 최소 네 가지가 겹쳐 있다:

1. **모델 크기**: 120B 대 110M — 1,000배 이상 차이. 대형 모델일수록 문맥을 더 길게/깊게 반영해
   중첩 기관명 같은 복잡한 경계를 더 잘 판단할 가능성이 있다 (직접 검증은 못 했음).
2. **test셋이 다름**: 공식은 AI Hub 비공개 hidden test셋(2,100건), 본 프로젝트는 공개
   Validation에서 직접 분리한 test셋(9,026건) — 동일 조건 비교가 아니므로 수치 비교는 참고용이다.
3. **태그 범위(스코프) 차이** — *6단계에서 새로 짚은 지점*: 공식 문서(official.md)의 벤치마크는
   R1~R8 규칙 적용을 전제로 설계됐다. 하지만 실제 공개 데이터에는 R1(인명)·R3(주소)·R7(기관명)만
   존재하고 R2(주민번호)·R4(날짜)·R8(사업자번호)는 단 한 건도 없다 (1단계 EDA). 즉 본 프로젝트가
   푸는 문제는 애초에 ORG가 90% 이상을 차지하는 3-클래스 문제이고, 공식 벤치마크가 채점됐을
   비공개 test셋은 더 다양한(어쩌면 더 어려운) 태그 구성이었을 가능성이 있다 — 태스크 난이도
   자체가 다를 수 있다는 뜻이며, 이는 모델 성능과 무관한 비교 조건의 차이다.
4. **경계(boundary) 오류**: 4단계 오분류 분석에서 확인한 대로, 전체 오류의 84%가 중첩 기관명의
   경계 판단 실패였다 (`국가정보원 심리전단` → `국가정보원`만 탐지 등). 이는 (1)의 모델 크기
   차이와도 관련이 있어 보이지만, 독립적으로도 본 모델의 가장 뚜렷한 약점이다.

**결론**: 0.9045 vs 0.9589라는 수치만으로 "경량 모델이 대형 모델의 94%까지 따라왔다"고
말하는 것은 과장이다. 정확히는 "조건이 다른 두 벤치마크에서, 경량 모델이 자체 test셋 기준
목표(0.80)를 크게 초과하는 F1을 냈고, 공식 수치와 방향은 비슷하지만 직접 비교는 불가능하다"가
정직한 요약이다.

## 3. 한계점

- **판결문 특화 표현**: 학습 데이터가 판결문에 한정되어 있어, 다른 도메인 텍스트에서는
  성능이 달라질 수 있다 (검증 안 함).
- **R1 사법인물 예외 처리**: 판사·검사 등 공직자 이름은 R1 익명화 대상에서 제외되는 규칙이
  원본 데이터에 있는데, 이 예외가 학습 라벨에 얼마나 일관되게 반영됐는지는 별도로 검증하지
  않았다.
- **512 토큰 truncation**: 전체 섹션의 13.29%가 512토큰을 초과해, 해당 섹션 후반부의
  annotation은 학습에 전혀 반영되지 못했다 (2단계).
- **태그 범위 축소 (PER·LOC·ORG 3종)**: 공개 데이터 자체에 R2/R4/R8(주민번호·날짜·사업자번호)
  annotation이 전혀 없어 애초에 학습이 불가능했다 — 모델의 한계가 아니라 데이터의 한계 (1단계).
- **span 위치 불일치 13.09%**: 그중 drift 유형(7.86%)은 필터링해 학습에서 제외했다 —
  원본 데이터의 약 8%를 활용하지 못한 셈이다 (1~2단계).
- **중첩 기관명 경계 오류**: 전체 오분류의 84%가 ORG이고, 대다수가 여러 단어가 겹친 기관명의
  경계를 짧게/넓게 잘못 잡는 패턴이다. PER F1(0.55)·LOC F1(0.82)이 ORG(0.92)보다 낮은 것과
  함께 본 모델의 가장 뚜렷한 약점이다 (4단계).
- **class weight로 PER/LOC 개선 실패 (2회 확인)**: naive 역빈도 가중치(Run4, F1 0.82로 급락)와
  O/ORG는 고정하고 PER/LOC만 완만히 올리는 capped 가중치(Run5, F1 0.89로 소폭 하락) 둘 다
  실패했다 — 가중치 강도의 문제가 아니라, 이 데이터·모델 조합에서는 loss 가중치 조정 자체가
  PER/LOC 성능을 끌어올리는 방향으로 작동하지 않는다는 뜻으로 보인다 (3단계).
- **원본 라벨 자체의 노이즈**: `다음과 같은`의 `다음`이 ORG로 잘못 라벨링된 사례처럼, 원본
  데이터셋의 규칙 기반 자동 라벨링 과정 자체에 오류가 있을 수 있다 — 이런 사례에서는 모델이
  틀린 게 아니라 정답 자체가 틀렸을 가능성이 있다 (4단계).
- **공식 벤치마크와의 태그 스코프 차이**: 위 2절 참조 — 직접적인 수치 비교가 불가능하다.
- **PER F1 미해결**: 4개 학습 실험(Run1~3, Run5) + 1개 실패 실험(Run4)을 거쳤지만 PER F1은
  0.47~0.55 구간에 머물렀다. 근본 원인이 클래스 불균형이 아니라는 것까지는 확인했지만,
  진짜 원인(엔티티 자체의 모호성? 절대적으로 적은 학습 샘플? 다른 요인?)은 이 프로젝트
  범위에서는 규명하지 못했다.

## 4. v2 방향 (이 프로젝트 이후 별도 태스크)

- **모델 교체**: KoBERT 또는 KLUE-RoBERTa로 백본을 바꿔 경계 판단·PER 인식이 개선되는지 확인.
- **태그 확장**: 날짜·주민번호가 포함된 다른 공개 데이터소스를 추가해 DAT/ID 태그를 실제로
  학습해본다 (본 프로젝트의 공개 데이터에는 아예 없었으므로).
- **PER 개선 — loss 가중치가 아닌 다른 접근**: 3단계에서 class weight 계열 2가지가 모두
  실패했으므로, 오버샘플링(PER/LOC이 포함된 섹션을 더 자주 보여주기), focal loss,
  또는 PER 전용 후처리 규칙(사전 매칭 등) 같은 다른 방향을 시도해볼 가치가 있다.
- **512 토큰 한계 대응**: 슬라이딩 윈도우로 긴 섹션을 겹치는 여러 조각으로 나눠 학습·추론하면
  truncation으로 누락되는 13.29%의 후반부 annotation을 살릴 수 있을 것이다.
- **라벨 노이즈 정량화**: `다음` 사례처럼 원본 라벨 자체가 틀린 사례가 얼마나 더 있는지
  체계적으로 샘플링해 비율을 추정하면, 모델 성능의 상한선을 더 정확히 알 수 있다.
- **Hugging Face Spaces 배포**: 5단계에서는 로컬 실행까지만 확인했다 — 실제 배포는 v2에서
  이어서 진행.